Disciplina: **Mineração de Dados**

Professor: **Wilson Castello Branco Neto**

Aula 12 - Exemplo 1: Regras de Associação

Nome: Wilson Castello Branco Neto

Leitura e apresentação do dataset com os dados das transações realizadas.

Dataset disponível em: https://www.kaggle.com/datasets/lissetteg/ecommerce-dataset


In [3]:
import pandas as pd
import numpy as np
from google.colab import drive

drive.mount('/content/gdrive')


Mounted at /content/gdrive


In [16]:
df = pd.read_csv('/content/gdrive/MyDrive/Ciencia_de_dados2/Aula 12data.csv')

df

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
...,...,...,...,...,...,...,...,...
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,12/9/2011 12:50,0.85,12680.0,France
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,12/9/2011 12:50,2.10,12680.0,France
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,12/9/2011 12:50,4.15,12680.0,France
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,12/9/2011 12:50,4.15,12680.0,France


Exclusão de todas as colunas desnecessárias. São utilizadas apenas o código da transação (**InvoiceNo**) e o código do produto (**StockCode**).

Além disto, se algum registro tiver com um destes atributos vazios ele será excluído.

In [ ]:
df=df[["InvoiceNo","StockCode"]]
df.dropna(inplace=True)

df

/tmp/ipython-input-119615714.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.dropna(inplace=True)


,InvoiceNo,StockCode
0,536365,85123A
1,536365,71053
2,536365,84406B
3,536365,84029G
4,536365,84029E
...,...,...
541904,581587,22613
541905,581587,22899
541906,581587,23254
541907,581587,23255


Agrupa os itens vendidos em cada transação em uma lista

In [ ]:
transacoes=df.groupby("InvoiceNo")["StockCode"].apply(list).reset_index(name='Items')

transacoes

,InvoiceNo,Items
0,536365,"[85123A, 71053, 84406B, 84029G, 84029E, 22752,..."
1,536366,"[22633, 22632]"
2,536367,"[84879, 22745, 22748, 22749, 22310, 84969, 226..."
3,536368,"[22960, 22913, 22912, 22914]"
4,536369,[21756]
...,...,...
25895,C581484,[23843]
25896,C581490,"[22178, 23144]"
25897,C581499,[M]
25898,C581568,[21258]


Cria uma lista de transacoes, sendo que em cada lista há uma sublista com os itens de cada transação. Apresenta as duas primeiras transações.

In [ ]:
transacoes=transacoes["Items"].tolist()

print(transacoes[0])
print(transacoes[1])



['85123A', '71053', '84406B', '84029G', '84029E', '22752', '21730']
['22633', '22632']


Codificação da lista de transações em valores binários.

In [ ]:
from mlxtend.preprocessing import TransactionEncoder

te = TransactionEncoder()
transacoes_codificadas = te.fit(transacoes).transform(transacoes)

print(transacoes_codificadas)

[[False False False ... False False False]
 [False False False ... False False False]
 [False False False ... False False False]
 ...
 [False False False ... False False False]
 [False False False ... False False False]
 [False False False ... False False False]]


Transformação das transações codificadas em um dataframe do pandas para facilitar sua manipulação.

In [ ]:
import pandas as pd

df = pd.DataFrame(transacoes_codificadas, columns=te.columns_)

df

,10002,10080,10120,10123C,10123G,10124A,10124G,10125,10133,10134,...,M,PADS,POST,S,gift_0001_10,gift_0001_20,gift_0001_30,gift_0001_40,gift_0001_50,m
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25895,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
25896,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
25897,False,False,False,False,False,False,False,False,False,False,...,True,False,False,False,False,False,False,False,False,False
25898,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


Geração dos itens frequentes, com suporte mínimo de 0.02.

In [ ]:
from mlxtend.frequent_patterns import apriori

itemsets = apriori(df,min_support=0.02, use_colnames=True)

print("Itemsets frequentes:")
print(itemsets)

Itemsets frequentes:
      support               itemsets
0    0.020193                (15036)
1    0.027181                (20685)
2    0.020541                (20711)
3    0.033668                (20712)
4    0.026023                (20713)
..        ...                    ...
215  0.022471        (85099B, 23203)
216  0.021197         (23301, 23300)
217  0.022896       (85099B, 85099C)
218  0.021042       (85099B, 85099F)
219  0.021197  (22697, 22698, 22699)

[220 rows x 2 columns]


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Criação das regras de associação com confiança mínima de 0.5 e transformação das regras geradas em um dataframe para melhorar a apresentação.

In [ ]:
from mlxtend.frequent_patterns import association_rules

regras = association_rules(itemsets, metric="confidence", min_threshold=0.5, num_itemsets=len(df) )
print("Regras de associação")
regras = pd.DataFrame(regras)
regras

Regras de associação


/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(20712),(85099B),0.033668,0.082432,0.020772,0.616972,7.484584,1.0,0.017997,2.395566,0.896578,0.217902,0.582562,0.434482
1,(20724),(22356),0.040541,0.029344,0.020309,0.500952,17.071930,1.0,0.019119,1.945018,0.981203,0.409657,0.485866,0.596529
2,(22356),(20724),0.029344,0.040541,0.020309,0.692105,17.071930,1.0,0.019119,3.116193,0.969884,0.409657,0.679096,0.596529
3,(20726),(20725),0.040039,0.062085,0.020541,0.513018,8.263168,1.0,0.018055,1.925976,0.915642,0.251775,0.480783,0.421932
4,(20727),(20725),0.050000,0.062085,0.025019,0.500386,8.059701,1.0,0.021915,1.877280,0.922027,0.287361,0.467314,0.451686
5,(22383),(20725),0.050425,0.062085,0.025598,0.507657,8.176813,1.0,0.022468,1.905003,0.924311,0.294536,0.475067,0.459985
6,(22384),(20725),0.042857,0.062085,0.023668,0.552252,8.895108,1.0,0.021007,2.094740,0.927321,0.291211,0.522614,0.466736
7,(21928),(85099B),0.031506,0.082432,0.021081,0.669118,8.117165,1.0,0.018484,2.773093,0.905327,0.227027,0.639392,0.462428
8,(21929),(85099B),0.033861,0.082432,0.020154,0.595211,7.220592,1.0,0.017363,2.266780,0.891701,0.209639,0.558846,0.419854
9,(21931),(85099B),0.046371,0.082432,0.028301,0.610325,7.403939,1.0,0.024479,2.354698,0.906995,0.281598,0.575317,0.476825


Criação de um novo dataframe apenas com as colunas mais relevantes para cada regra.

In [ ]:
resumo = regras[['antecedents','consequents','support','confidence']]
resumo

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


,antecedents,consequents,support,confidence
0,(20712),(85099B),0.020772,0.616972
1,(20724),(22356),0.020309,0.500952
2,(22356),(20724),0.020309,0.692105
3,(20726),(20725),0.020541,0.513018
4,(20727),(20725),0.025019,0.500386
5,(22383),(20725),0.025598,0.507657
6,(22384),(20725),0.023668,0.552252
7,(21928),(85099B),0.021081,0.669118
8,(21929),(85099B),0.020154,0.595211
9,(21931),(85099B),0.028301,0.610325
